# Fine-tune QLoRA — **um expert por vez**, DeepSeek → depois Qwen

Treina um adapter LoRA por expert em cima de um modelo-base de código, usando as
**mesmas lições** que alimentam o router/KB do pyaxon.

**Ordem:** rode inteiro com `BASE = "deepseek"`. Depois volte na célula 3, troque pra
`BASE = "qwen"` e rode de novo. Os dois adapters convivem no Drive
(`axon_lora/deepseek/go/` e `axon_lora/qwen/go/`) e a célula 8 compara os dois.

**Checkpoints:** o treino grava direto no Drive a cada 50 passos. Se a sessão cair,
rode a mesma célula de novo — ele retoma do último checkpoint em vez de recomeçar.

> `Ambiente de execução → Alterar tipo de ambiente → GPU (T4)` — obrigatório aqui.

## 1. Instalar (Unsloth = QLoRA rápido e leve)

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets

import torch
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 2. Clonar o repo (traz os helpers e o mapa de experts)

In [ ]:
REPO = "https://github.com/geraldogrise/axon-llm.git"

import os, sys, json
if not os.path.isdir("axon-llm"):
    !git clone --depth 1 $REPO axon-llm
sys.path.insert(0, "/content/axon-llm/notebooks")
import axon_colab as ac

## 3. Escolher o modelo-base e o expert

**Primeira passada:** `deepseek`. **Segunda:** `qwen`. Um de cada vez — 7B em 4-bit
já ocupa quase toda a T4, não dá pra carregar os dois juntos.

In [ ]:
BASE   = "deepseek"   # "deepseek" -> deepseek-coder-6.7b | "qwen" -> Qwen2.5-Coder-7B
                      # "qwen-3b"  -> se a T4 estourar memória
EXPERT = "go"         # mesmo nome do outro notebook

MODEL   = ac.BASES[BASE]
MAX_LEN = 2048
_, _, _, _, ROTULO = ac.check(EXPERT)

CKPT = ac.drive_dir("axon_ckpt", BASE, EXPERT)   # checkpoints (retomável)
LORA = ac.lora_dir(BASE, EXPERT)                 # adapter final

print(f"base       : {MODEL}")
print(f"expert     : {EXPERT} ({ROTULO})")
print(f"checkpoints: {CKPT}")
print(f"adapter    : {LORA}")

## 4. Carregar o modelo em 4-bit + adaptadores LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=MAX_LEN, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0.0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)

## 5. Montar o dataset a partir das lições

Clona a branch de dados do expert e transforma cada `.md` em par pergunta/resposta.
A pergunta cita **linguagem + família + subsetor** (que vêm do path da lição), não só
o título — é o que ensina o modelo a *responder sobre o assunto* em vez de completar
o texto da lição. Lições longas são quebradas nas seções `##` pra caber no contexto.

In [ ]:
from datasets import Dataset

dados = ac.fetch_data(EXPERT)
exemplos = ac.sft_examples(dados, tokenizer, ROTULO, max_chars=6000)
print(f"{len(exemplos)} exemplos de treino")
print("---- amostra ----")
print(exemplos[0]["text"][:600])

ds = Dataset.from_list(exemplos).train_test_split(test_size=0.05, seed=42)
print(ds)

## 6. Treinar (QLoRA) — com checkpoint no Drive

`save_steps=50` grava no Drive durante o treino. Se a sessão cair, **rode esta célula
de novo**: ela acha o último `checkpoint-<n>` e retoma dali.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=ds["train"], eval_dataset=ds["test"],
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_LEN,
        per_device_train_batch_size=1, gradient_accumulation_steps=8,   # 7B na T4
        num_train_epochs=2, warmup_steps=10, learning_rate=2e-4,
        optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="cosine",
        logging_steps=10, seed=42, report_to="none",
        output_dir=CKPT, save_steps=50, save_total_limit=2,
    ),
)

retomar = ac.ultimo_checkpoint(CKPT)
print(f"retomando de {retomar}" if retomar else "treino do zero")
trainer.train(resume_from_checkpoint=retomar)

## 7. Salvar o adapter e a métrica (pra comparar os dois modelos depois)

In [ ]:
model.save_pretrained(LORA)
tokenizer.save_pretrained(LORA)

metrica = trainer.evaluate()
metrica.update({"base": BASE, "modelo": MODEL, "expert": EXPERT,
                "exemplos": len(exemplos)})
with open(os.path.join(LORA, "metrica.json"), "w", encoding="utf-8") as f:
    json.dump(metrica, f, ensure_ascii=False, indent=2)

print(json.dumps(metrica, ensure_ascii=False, indent=2))
print("adapter salvo em:", LORA)

## 8. Testar

In [ ]:
FastLanguageModel.for_inference(model)

def responder(pergunta, max_new_tokens=400):
    ids = tokenizer.apply_chat_template([{"role": "user", "content": pergunta}],
                                        tokenize=True, add_generation_prompt=True,
                                        return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens,
                         temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

print(responder(f"Em {ROTULO}, como faço tratamento de erros de forma idiomática?"))

## 9. Trocar de modelo-base

Volte na **célula 3**, ponha `BASE = "qwen"` e rode 3 → 8 de novo.

> Reinicie o ambiente (`Ambiente de execução → Reiniciar sessão`) antes, pra liberar a
> VRAM do primeiro modelo.

Com os dois treinados, esta célula compara lado a lado:

In [ ]:
import glob

print(f"{'base':<10} {'expert':<12} {'eval_loss':>10} {'exemplos':>9}")
for fp in sorted(glob.glob("/content/drive/MyDrive/axon_lora/*/*/metrica.json")):
    m = json.load(open(fp, encoding="utf-8"))
    print(f"{m['base']:<10} {m['expert']:<12} {m.get('eval_loss', float('nan')):>10.4f} "
          f"{m.get('exemplos', 0):>9}")

## 10. Juntar as duas metades (router + KB + LoRA)

O expert do outro notebook escolhe **onde** procurar e recupera a lição certa; o LoRA
responde. Isso é o que evita o modelo inventar API que não existe.

Precisa do `_axon.so` compilado — rode antes as células 1–3 do `train_expert_colab.ipynb`
(ou a célula abaixo, que é a mesma compilação).

In [ ]:
import pyaxon as ax   # exige o build do train_expert_colab.ipynb

expert_dir = f"/content/drive/MyDrive/axon_experts/{ac.SAIDA[EXPERT]}"
router = ax.modular.ModularRouter.load(os.path.join(expert_dir, "router"))
kb = ax.vindex.SparseKB.load(os.path.join(expert_dir, "kb.sparse.json.gz"))

def responder_com_contexto(pergunta, k=3):
    caminho = router.route(pergunta)
    trechos = [t[0] for t in kb.retrieve(pergunta, path_prefix=caminho, top_k=k)]
    contexto = "\n\n---\n\n".join(trechos)
    prompt = (f"Use o material abaixo pra responder. Se ele não cobrir a pergunta, "
              f"diga isso em vez de inventar.\n\n{contexto}\n\nPergunta: {pergunta}")
    return " > ".join(caminho), responder(prompt)

rota, resp = responder_com_contexto("como uso canais com select?")
print(f"[rota: {rota}]\n{resp}")